<a href="https://colab.research.google.com/github/Aseel07117/AI-Project/blob/main/task1_base_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 – Testing the base models (M2)
CoNaLa curated · `Qwen2.5-0.5B-Instruct` vs `Qwen2.5-Coder-0.5B-Instruct`

Same system prompt and generation settings for both models (greedy decoding, max 128 new tokens). Keep them identical in Task 2.

**Runtime → Change runtime type → GPU (T4)**

In [ ]:
!pip -q install datasets transformers accelerate

## 1. Data preparation (same as M1)

In [ ]:
from datasets import load_dataset

url = ("https://huggingface.co/datasets/neulab/conala/"
       "resolve/refs%2Fconvert%2Fparquet/curated/")
ds = load_dataset("parquet", data_files={
    "train": url + "train/0000.parquet",
    "test":  url + "test/0000.parquet",
})
train, test = ds["train"], ds["test"]
split = train.train_test_split(test_size=0.1, seed=42)
train, val = split["train"], split["test"]
print("train:", len(train), " validation:", len(val), " test:", len(test))

0000.parquet:   0%|          | 0.00/217k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/48.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

train: 2141  validation: 238  test: 500


## 2. Prompt, metrics and answer cleaning

In [ ]:
import ast, re

SYSTEM_PROMPT = ("You are a Python coding assistant. Reply with exactly one line of "
                 "Python code that answers the request. Do not add explanations or markdown.")

def build_messages(intent, snippet=None):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": intent}]
    if snippet is not None:                       # used in Task 2 (training rows)
        msgs.append({"role": "assistant", "content": snippet})
    return msgs

def clean_answer(text):
    """Handle ```python fences and empty answers. Same function for all 4 models."""
    text = (text or "").strip()
    m = re.search(r"```(?:python|py)?\s*\n?(.*?)```", text, flags=re.S | re.I)
    if m:
        text = m.group(1)
    return text.strip()

def is_valid(code):
    if not code.strip():                          # empty answer = invalid
        return False
    try:
        ast.parse(code)
        return True
    except (SyntaxError, ValueError):
        return False

def normalize(text):
    return " ".join(text.split())

def exact_match(pred, ref):
    return normalize(pred) == normalize(ref)

def scores(preds, refs):
    em = 100 * sum(exact_match(p, r) for p, r in zip(preds, refs)) / len(refs)
    va = 100 * sum(is_valid(p) for p in preds) / len(preds)
    return em, va

## 3. Generation (greedy, max 128 new tokens)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def generate_answers(model_name, questions, batch_size=16, adapter_path=None):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
    if adapter_path:                              # Task 2: load the LoRA adapter
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    prompts = [tok.apply_chat_template(build_messages(q), tokenize=False, add_generation_prompt=True)
               for q in questions]
    answers = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i+batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=128, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        new = out[:, enc["input_ids"].shape[1]:]
        answers += [clean_answer(t) for t in tok.batch_decode(new, skip_special_tokens=True)]
        print(f"{min(i+batch_size, len(prompts))}/{len(prompts)}", end="\r")
    del model
    torch.cuda.empty_cache()
    return answers

## 4. Run both base models on the 500 test questions

In [ ]:
import pandas as pd

GENERAL = "Qwen/Qwen2.5-0.5B-Instruct"
CODER   = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

questions, refs = test["intent"], test["snippet"]
general_base = generate_answers(GENERAL, questions)
coder_base   = generate_answers(CODER, questions)

df = pd.DataFrame({"question": questions, "reference": refs,
                   "general_base": general_base, "code_base": coder_base})
df.to_csv("task1_base_answers.csv", index=False)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (9 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## 5. Metrics

In [ ]:
rows = []
for name, preds in [("Qwen2.5-0.5B-Instruct", general_base), ("Qwen2.5-Coder-0.5B-Instruct", coder_base)]:
    em, va = scores(preds, refs)
    rows.append({"Model": name, "Stage": "Before fine-tuning",
                 "Exact match (%)": round(em, 1), "Valid Python (%)": round(va, 1)})
results = pd.DataFrame(rows)
results.to_csv("task1_results.csv", index=False)
results

,Model,Stage,Exact match (%),Valid Python (%)
0,Qwen2.5-0.5B-Instruct,Before fine-tuning,0.0,93.2
1,Qwen2.5-Coder-0.5B-Instruct,Before fine-tuning,0.2,86.6


## 6. Look at a few answers (sanity check)

In [ ]:
df.head(10)

,question,reference,general_base,code_base
0,How can I send a signal from a python program?,"os.kill(os.getpid(), signal.SIGUSR1)","import signal\n\ndef handler(signum, frame):\n...",import os\n\n# Define the signal to be sent\ns...
1,Decode Hex String in Python 3,bytes.fromhex('4a4b4c').decode('utf-8'),def decode_hex(hex_string):\n return hex_st...,"hex_string = ""1A3F"" # Example hexadecimal str..."
2,check if all elements in a list are identical,all(x == myList[0] for x in myList),def all_elements_identical(lst):\n return l...,"all(x == y for x, y in zip(list1, list2))"
3,Format string dynamically,"print('%*s : %*s' % (20, 'Python', 20, 'Very G...","formatted_string = ""Hello, {name}! You have {c...","formatted_string = f""Hello, {name}! You have {..."
4,How to convert a string from CP-1251 to UTF-8?,d.decode('cp1251').encode('utf8'),import codecs\n\n# Convert CP-1251 encoded str...,import codecs\n\ndef convert_cp1251_to_utf8(in...
5,How I can get rid of None values in dictionary?,"res = {k: v for k, v in list(kwargs.items()) i...",def remove_none_values(d):\n return {k: v f...,# Define a dictionary with some None values\nm...
6,How I can get rid of None values in dictionary?,"res = dict((k, v) for k, v in kwargs.items() i...",def remove_none_values(d):\n return {k: v f...,# Define a dictionary with some None values\nm...
7,Python: how to get the final output of multipl...,subprocess.check_output('ps -ef | grep somethi...,import subprocess\n\n# List of command lines\n...,import subprocess\n\n# Execute multiple system...
8,splitting and concatenating a string,""""""""""""".join(['a', 'b', 'c'])","string = ""Hello, world!""\nnew_string = string....","string = ""Hello, World!"""
9,Finding the intersection between two series in...,pd.Series(list(set(s1).intersection(set(s2)))),import pandas as pd\n\n# Example dataframes\nd...,```python\nimport pandas as pd\n\n# Sample dat...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')